# Tome 15, Appendix L: Open Doors in Algebra (Python companion)

Exercises first, solutions at the end. We build polynomials with Galois group $S_n$ as in Theorem 12.3 and watch their Frobenius cycle types, search
for small Mahler measures (Figure 12.2), look inside the proof of Theorem 12.6, and test Theorem 12.8 and Theorem 12.9 on
maps over finite fields. We use `numpy` and `sympy`.

*Before each solution, predict what the output will look like.*

In [1]:
import itertools, math, random, time, warnings
from collections import Counter
import numpy as np
import sympy as sp
warnings.filterwarnings('ignore')
X = sp.symbols('X')

## Exercises

**P12.1 (★★) $S_n$ by the Chinese remainder theorem.** For $2\le n\le10$, build the polynomial $f=-15f_2+10f_3+6f_5$ of Theorem 12.3, with
coefficients reduced to $[-15,15)$, and check its factorisations modulo $2$, $3$, $5$ and that $30$ is prime to its discriminant. Check Example 12.4.

**P12.2 (★★) Chebotarev at work.** For the sextic of Example 12.4, record the degrees of the irreducible factors modulo every prime
$p<20\,000$ not dividing the discriminant, and compare the frequencies of the cycle types with the proportions of permutations of $S_6$ of each type.

**P12.3 (★★) Small Mahler measures.** For each degree $d\le12$, find the smallest Mahler measure $>1$ among the polynomials with coefficients in
$\{-1,0,1\}$, leading coefficient $1$ and constant term $\pm1$, separately for reciprocal and non-reciprocal polynomials (Figure 12.2).

**P12.4 (★) Inside the proof of Theorem 12.6.** For $P=X^3-X-1$, compute the first coefficients of $G=P/P^*$, $g$ and $h$, and check the
inequalities $|c_0||q_k|\le|b_k|+|c_k|$ and $|b_k|,|c_k|\le1-|c_0|^2$. Do the same for Lehmer's polynomial: what goes wrong?

**P12.5 (★★) Quadratic maps over finite fields.** Draw random quadratic maps $\mathbb F_p^n\to\mathbb F_p^n$ for $(p,n)=(3,2)$, $(5,2)$, $(7,2)$ and $(3,3)$. Among those whose
Jacobian determinant vanishes nowhere on $\mathbb F_p^n$, check that all are bijective (Theorem 12.8). How many of the others are bijective? Check
also the two counterexamples of Exercise 12.7.

**P12.6 (★) Exercises 12.1, 12.2 and 12.9.** Check the minimal polynomial of $2\cos\frac{2\pi}{11}$, the factorisations of
$X^4-X-1$, and the inverse of the map of Exercise 12.9.

## Solutions

### P12.1
We take for each $f_p$ the first suitable irreducible polynomials in lexicographic order, as in the proof of Theorem 12.3.

In [2]:
from sympy.polys.galoistools import gf_irreducible_p
from sympy.polys.domains import ZZ

def irr(p, d):
    for cs in itertools.product(range(p), repeat=d):
        c = [1] + list(cs)
        if d > 1 and c[-1] == 0: continue
        if gf_irreducible_p(c, p, ZZ): return sp.Poly(c, X)

def build(n):
    f2 = irr(2, n)
    f3 = irr(3, n - 1) * sp.Poly(X, X) if n >= 3 else sp.Poly(X * (X + 1), X)
    m = n - 2
    if m == 0: r = sp.Poly(1, X)
    elif m == 1: r = sp.Poly(X, X)
    elif m % 2 == 1: r = irr(5, m)
    elif m == 2: r = sp.Poly(X * (X + 1), X)
    else: r = irr(5, m - 1) * sp.Poly(X, X)
    f = -15 * f2 + 10 * f3 + 6 * (irr(5, 2) * r)
    c = [((a + 15) % 30) - 15 for a in f.all_coeffs()]
    return sp.Poly(c, X)

def degs(f, p):
    return sorted(sp.Poly(g, X).degree() for g, e in sp.factor_list(f.as_expr(), modulus=p)[1] for _ in range(e))

for n in range(2, 11):
    f = build(n)
    d = sp.discriminant(f.as_expr(), X)
    print('n=%2d' % n, f.as_expr(), '| mod 2,3,5:', degs(f, 2), degs(f, 3), degs(f, 5), '| gcd(disc,30) =', math.gcd(int(d), 30))
g6 = sp.Poly(X**6 - 12*X**4 + 6*X**3 + 2*X**2 + 7*X - 15, X)
print('Example:', build(6) == g6, sp.factorint(sp.discriminant(g6.as_expr(), X)))

n= 2 X**2 - 5*X - 3 | mod 2,3,5: [2] [1, 1] [2] | gcd(disc,30) = 1
n= 3 X**3 + 7*X - 15 | mod 2,3,5: [3] [1, 2] [1, 2] | gcd(disc,30) = 1
n= 4 X**4 + 6*X**3 + 2*X**2 + 7*X - 15 | mod 2,3,5: [4] [1, 3] [1, 1, 2] | gcd(disc,30) = 1
n= 5 X**5 - 12*X**3 + X**2 + 2*X - 3 | mod 2,3,5: [5] [1, 4] [2, 3] | gcd(disc,30) = 1
n= 6 X**6 - 12*X**4 + 6*X**3 + 2*X**2 + 7*X - 15 | mod 2,3,5: [6] [1, 5] [1, 2, 3] | gcd(disc,30) = 1
n= 7 X**7 + 12*X**5 - 6*X**3 - 14*X**2 - 7*X - 3 | mod 2,3,5: [7] [1, 6] [2, 5] | gcd(disc,30) = 1
n= 8 X**8 + 12*X**6 + 9*X**4 + X**3 - 12*X**2 - 13*X - 15 | mod 2,3,5: [8] [1, 7] [1, 2, 5] | gcd(disc,30) = 1
n= 9 X**9 + 12*X**7 - 14*X**3 + 6*X**2 - 13*X - 3 | mod 2,3,5: [9] [1, 8] [2, 7] | gcd(disc,30) = 1
n=10 X**10 + 12*X**8 - 4*X**4 + X**3 + 12*X**2 - 8*X - 15 | mod 2,3,5: [10] [1, 9] [1, 2, 7] | gcd(disc,30) = 1
Example: True {951221: 1, 6107273: 1}


Every line shows the three patterns of the proof: $[n]$ modulo $2$, $[1,n-1]$ modulo $3$, and a quadratic factor with odd-degree factors modulo $5$.

### P12.2
The proportion of permutations of $S_6$ with a given cycle type is $1/\prod_j j^{m_j}m_j!$, where $m_j$ is the number of cycles of length $j$.

In [3]:
def ctype_prop(parts):
    m = Counter(parts)
    return 1 / math.prod(j**k * math.factorial(k) for j, k in m.items())

D = int(sp.discriminant(g6.as_expr(), X))
cnt = Counter()
t0 = time.time()
for p in sp.primerange(2, 20000):
    if D % p == 0: continue
    cnt[tuple(degs(g6, p))] += 1
tot = sum(cnt.values())
print('%d primes, %.0f s' % (tot, time.time() - t0))
print('cycle type       observed   S_6')
for t in sorted(cnt, key=lambda t: (len(t), t)):
    print('%-16s %8.4f %8.4f' % (t, cnt[t] / tot, ctype_prop(t)))
print('number of cycle types seen:', len(cnt), '(S_6 has 11)')

2262 primes, 7 s
cycle type       observed   S_6
(6,)               0.1706   0.1667
(1, 5)             0.2034   0.2000
(2, 4)             0.1242   0.1250
(3, 3)             0.0535   0.0556
(1, 1, 4)          0.1260   0.1250
(1, 2, 3)          0.1596   0.1667
(2, 2, 2)          0.0208   0.0208
(1, 1, 1, 3)       0.0548   0.0556
(1, 1, 2, 2)       0.0601   0.0625
(1, 1, 1, 1, 2)    0.0243   0.0208
(1, 1, 1, 1, 1, 1)   0.0027   0.0014
number of cycle types seen: 11 (S_6 has 11)


All eleven cycle types of $S_6$ appear, with the frequencies predicted by Chebotarev's theorem (Tome~12, Chapter~8) for the group $S_6$: another
confirmation that the group is the whole symmetric group, since a proper subgroup would miss some types.

### P12.3
We compute all roots at once, as eigenvalues of companion matrices. Repeated roots on the unit circle are computed only to about $10^{-8}$, so we call a
measure ``$>1$'' only above $1.001$; the next value after $1$ is far above that.

In [4]:
def search(dmax):
    out = {}
    for d in range(2, dmax + 1):
        mids = np.array(list(itertools.product((-1, 0, 1), repeat=d - 1)), dtype=float)
        best = {'rec': (9, None), 'non': (9, None)}
        for c0 in (1, -1):
            coef = np.hstack([np.ones((len(mids), 1)), mids, np.full((len(mids), 1), c0)])
            rev = coef[:, ::-1]
            isrec = np.all(coef == rev, axis=1) | np.all(coef == -rev, axis=1)
            Cm = np.zeros((len(coef), d, d)); Cm[:, 0, :] = -coef[:, 1:]
            Cm[:, np.arange(1, d), np.arange(d - 1)] = 1
            M = np.prod(np.maximum(1, np.abs(np.linalg.eigvals(Cm))), axis=1)
            for key, mask in (('rec', isrec), ('non', ~isrec)):
                m = np.where(mask & (M > 1.001), M, 9)
                i = int(np.argmin(m))
                if m[i] < best[key][0]: best[key] = (m[i], coef[i].astype(int).tolist())
        out[d] = best
        print('d=%2d  reciprocal %s  non-reciprocal %.6f %s' % (d, ('%.6f %s' % best['rec']) if best['rec'][1] else '   none', *best['non']))
    return out
res = search(12)

d= 2  reciprocal    none  non-reciprocal 1.618034 [1, -1, -1]
d= 3  reciprocal    none  non-reciprocal 1.324718 [1, -1, 0, 1]
d= 4  reciprocal 1.722084 [1, -1, -1, -1, 1]  non-reciprocal 1.324718 [1, 0, -1, -1, 1]
d= 5  reciprocal 1.883204 [1, -1, -1, -1, -1, 1]  non-reciprocal 1.324718 [1, 0, 0, 0, 1, 1]
d= 6  reciprocal 1.401268 [1, 0, -1, -1, -1, 0, 1]  non-reciprocal 1.324718 [1, 0, 1, 0, 0, 0, -1]
d= 7  reciprocal 1.401268 [1, 1, -1, 0, 0, -1, 1, 1]  non-reciprocal 1.324718 [1, -1, 0, 0, -1, 1, -1, 1]
d= 8  reciprocal 1.280638 [1, 0, 0, -1, -1, -1, 0, 0, 1]  non-reciprocal 1.324718 [1, -1, 0, -1, 1, -1, 0, 0, 1]


d= 9  reciprocal 1.280638 [1, 1, 0, 1, 0, 0, 1, 0, 1, 1]  non-reciprocal 1.324718 [1, -1, 0, 0, 1, 0, 0, -1, 0, 1]


d=10  reciprocal 1.176281 [1, -1, 0, 1, -1, 1, -1, 1, 0, -1, 1]  non-reciprocal 1.324718 [1, 0, 0, 0, 0, 0, 0, 0, 1, 0, -1]


d=11  reciprocal 1.176281 [1, 0, -1, 1, 0, 0, 0, 0, 1, -1, 0, 1]  non-reciprocal 1.324718 [1, -1, 1, -1, 1, -1, 0, 0, 0, 0, -1, 1]


d=12  reciprocal 1.176281 [1, 0, 0, 0, 0, -1, -1, -1, 0, 0, 0, 0, 1]  non-reciprocal 1.324718 [1, 0, 0, -1, 0, 1, 1, 0, 0, -1, 1, 0, 1]


The non-reciprocal minimum is $1.324718$ from degree $3$ on, Smyth's constant, never the $1.2808$ of Theorem 12.6: our bound is true but not
sharp. The reciprocal minimum drops to Lehmer's $1.176281$ at degree $10$ (the minimiser printed is $L(-X)$, which has the same measure) and stays there.

### P12.4
We expand rational functions into power series by exact division of polynomial coefficient lists.

In [5]:
def series(num, den, N):
    # power series of num/den (coefficient lists, lowest degree first), den[0] != 0
    num = list(num) + [0] * N; out = []
    for i in range(N):
        c = num[i] / den[0]; out.append(c)
        for j, dj in enumerate(den):
            if i + j < len(num): num[i + j] -= c * dj
    return np.array(out)

def inspect(P, N=8):
    P = np.array(P, dtype=float)                # highest degree first
    roots = np.roots(P)
    low = P[::-1]                               # P, lowest degree first
    Pstar = P                                   # P*(z) = z^d P(1/z): reversed coefficients, lowest first
    G = series(low, Pstar, N)
    num_g, den_g, num_h, den_h = np.array([1+0j]), np.array([1+0j]), np.array([1+0j]), np.array([1+0j])
    for a in roots:
        if abs(a) < 1: num_g = np.convolve(num_g, [-a, 1]); den_g = np.convolve(den_g, [1, -a])
        elif abs(a) > 1: num_h = np.convolve(num_h, [1, -a]); den_h = np.convolve(den_h, [-a, 1])
    g, h = series(num_g, den_g, N), series(num_h, den_h, N)
    q = np.round(G.real).astype(int)
    print('G = P/P* coefficients:', q.tolist())
    k = next(i for i in range(1, N) if q[i] != 0)
    c0 = abs(h[0]); M = np.prod([max(1, abs(a)) for a in roots])
    print('M = %.6f, |c_0| = %.6f = 1/M, first k = %d, q_k = %d' % (M, c0, k, q[k]))
    print('|c0 q_k| = %.4f <= |b_k| + |c_k| = %.4f + %.4f;  1 - c0^2 = %.4f' % (c0 * abs(q[k]), abs(g[k]), abs(h[k]), 1 - c0**2))
    print('check g = hG:', np.allclose(g, np.convolve(h, G)[:N]))

inspect([1, 0, -1, -1])
G_L = series([1, 1, 0, -1, -1, -1, -1, -1, 0, 1, 1][::-1], [1, 1, 0, -1, -1, -1, -1, -1, 0, 1, 1], 8)
print('Lehmer: P/P* =', np.round(G_L, 6).tolist())

G = P/P* coefficients: [-1, -1, -1, -1, -2, -2, -3, -4]
M = 1.324718, |c_0| = 0.754878 = 1/M, first k = 1, q_k = -1
|c0 q_k| = 0.7549 <= |b_k| + |c_k| = 0.3247 + 0.4302;  1 - c0^2 = 0.4302
check g = hG: True
Lehmer: P/P* = [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


For $x^3-x-1$ (with $k=1$) the chain is tight in two places: $|c_0q_1|=|b_1|+|c_1|$, and $|c_1|=1-|c_0|^2$ exactly. All the slack is in
$|b_1|=0.3247<0.4302$: the lemma bounds $b$ and $c$ separately, and Smyth's sharper argument gains by controlling them together. For Lehmer's polynomial $P=P^*$, the series is the
constant $1$: there is no first non-zero coefficient, and the proof has nothing to work with.

### P12.5
A quadratic map is given by a constant, a linear part and a quadratic part for each component; we evaluate it on all $p^n$ points.

In [6]:
def random_quadratic(p, n, rng):
    mons = [m for m in itertools.product(range(3), repeat=n) if sum(m) <= 2]
    return [{m: rng.randrange(p) for m in mons} for _ in range(n)]

def evaluate(F, x, p):
    return tuple(sum(c * math.prod(xi**e for xi, e in zip(x, m)) for m, c in comp.items()) % p for comp in F)

def jac_det(F, x, p):
    n = len(F)
    J = []
    for comp in F:
        row = []
        for j in range(n):
            s = 0
            for m, c in comp.items():
                if m[j] == 0: continue
                mm = list(m); mm[j] -= 1
                s += c * m[j] * math.prod(xi**e for xi, e in zip(x, mm))
            row.append(s % p)
        J.append(row)
    return idet(J) % p

def idet(A):
    # integer determinant by cofactor expansion (n <= 3 here)
    if len(A) == 1: return A[0][0]
    return sum((-1)**j * A[0][j] * idet([r[:j] + r[j+1:] for r in A[1:]]) for j in range(len(A)))

def experiment(p, n, trials, seed=1):
    rng = random.Random(seed)
    pts = list(itertools.product(range(p), repeat=n))
    stats = Counter()
    for _ in range(trials):
        F = random_quadratic(p, n, rng)
        nowhere0 = all(jac_det(F, x, p) for x in pts)
        bij = len(set(evaluate(F, x, p) for x in pts)) == len(pts)
        stats[(nowhere0, bij)] += 1
    return stats

for p, n, T in ((3, 2, 20000), (5, 2, 40000), (7, 2, 40000), (3, 3, 100000)):
    st = experiment(p, n, T)
    print('p=%d n=%d: Jacobian nowhere zero: %d maps, all bijective: %s;  others: %d, of which bijective: %d'
          % (p, n, st[(True, True)] + st[(True, False)], st[(True, False)] == 0, st[(False, True)] + st[(False, False)], st[(False, True)]))
p = 5
print('x - x^p on F_5:', [(x - x**p) % p for x in range(p)])
print('x + x^2 on F_2:', [(x + x * x) % 2 for x in range(2)])

p=3 n=2: Jacobian nowhere zero: 152 maps, all bijective: True;  others: 19848, of which bijective: 0


p=5 n=2: Jacobian nowhere zero: 39 maps, all bijective: True;  others: 39961, of which bijective: 0


p=7 n=2: Jacobian nowhere zero: 14 maps, all bijective: True;  others: 39986, of which bijective: 0


p=3 n=3: Jacobian nowhere zero: 2 maps, all bijective: True;  others: 99998, of which bijective: 0
x - x^p on F_5: [0, 0, 0, 0, 0]
x + x^2 on F_2: [0, 0]


Every map with nowhere-vanishing Jacobian is a bijection, as Theorem 12.8 and Theorem 12.9 predict; maps whose Jacobian vanishes
somewhere are rarely bijective. A nowhere-vanishing Jacobian is rare among random quadratic maps (it is a strong condition), so we need many trials to see a
few, and in our samples no map whose Jacobian vanishes somewhere was bijective. In characteristic $p$, $x-x^p$ is the zero function on $\mathbb F_p$ although its derivative is $1$, and $x+x^2$ kills Wang's theorem in characteristic
$2$.

### P12.6

In [7]:
eta = [2 * math.cos(2 * math.pi * k / 11) for k in range(1, 6)]
print('minimal polynomial of 2cos(2pi/11):', np.round(np.poly(eta), 10).tolist())
h4 = sp.Poly(X**4 - X - 1, X)
for p in (2, 7, 17):
    print('X^4 - X - 1 mod %2d:' % p, sp.factor_list(h4.as_expr(), modulus=p)[1])
print('disc =', sp.discriminant(h4.as_expr(), X), 'prime:', sp.isprime(283))
x, y, u, v = sp.symbols('x y u v')
F = sp.Matrix([x + y**2, y + (x + y**2)**2])
Finv = sp.Matrix([u - (v - u**2)**2, v - u**2])
print('det J_F =', sp.simplify(F.jacobian([x, y]).det()))
print('F^{-1}(F(x,y)) =', sp.simplify(Finv.subs({u: F[0], v: F[1]}, simultaneous=True)).T)

minimal polynomial of 2cos(2pi/11): [1.0, 1.0, -4.0, -3.0, 3.0, 1.0]
X^4 - X - 1 mod  2: [(X**4 + X + 1, 1)]
X^4 - X - 1 mod  7: [(X - 3, 1), (X**3 + 3*X**2 + 2*X - 2, 1)]
X^4 - X - 1 mod 17: [(X + 2, 1), (X + 5, 1), (X**2 - 7*X + 5, 1)]
disc = -283 prime: True
det J_F = 1
F^{-1}(F(x,y)) = Matrix([[x, y]])


The minimal polynomial is $X^5+X^4-4X^3-3X^2+3X+1$, the factorisations are those of Exercise 12.2, and $F^{-1}\circ F$ is the identity.